# 🏛️ Lab 1: SQL Data Profiling & Extraction

**Welcome, Kevin!** (Lab Instructor)

## Background

The Driver and Vehicle Licensing Authority (DVLA) Ghana maintains a legacy vehicle registration 
database that has accumulated data quality issues over years of manual entry, system migrations, 
and inconsistent data capture across 20+ regional branch offices. Before any modernization or 
analytical reporting can be trusted, we must first **profile** this data to understand the scale 
and nature of the problems.

## What You Will Learn

In this lab, you will use **DuckDB** — a fast, in-process SQL database engine — to:

1. Load raw CSV datasets into in-memory tables
2. Inspect table schemas and row counts
3. Identify exact duplicate records
4. Profile missing identification fields
5. Detect payment anomalies (zero/negative amounts)
6. Find orphan payment records with no matching vehicle registration
7. Summarize anomalies by regional branch office

## Datasets

| File | Description |
|---|---|
| `../raw_data/legacy_vehicle_registry.csv` | 50,000+ vehicle registration records |
| `../raw_data/payment_transaction_log.csv` | 50,000+ payment transaction records |

---

## Step 1: Install and Import DuckDB

**📋 What you will do:**
Install the DuckDB Python package (if not already installed) and import it.

**🔧 Code to type:**
```python
# Install DuckDB if needed (only runs once)
!pip install duckdb --quiet

# Import the DuckDB library
import duckdb

print('DuckDB version:', duckdb.__version__)
print('✅ DuckDB ready!')
```

**📖 Line-by-line explanation:**
- `!pip install duckdb --quiet`: The `!` prefix runs a shell command from inside Jupyter. 
  `--quiet` suppresses verbose output. This installs DuckDB if it's not already available.
- `import duckdb`: Loads the DuckDB library into your Python session. DuckDB is an 
  **in-process** database — unlike PostgreSQL or MySQL, it runs entirely inside your Python 
  process with zero external server setup.
- `duckdb.__version__`: Prints the installed version for debugging.

**💡 Why this matters for DVLA:**
DuckDB can query CSV files directly without needing to set up a database server. This makes it 
ideal for quick data profiling in field offices where IT infrastructure may be limited.

In [ ]:
# Step 1: Install and Import DuckDB
# Type your code below


## Step 2: Create an In-Memory Connection and Load CSV Files

**📋 What you will do:**
Create a DuckDB in-memory database connection and load both raw CSV files into tables.

**🔧 Code to type:**
```python
# Create an in-memory DuckDB connection
con = duckdb.connect()

# Load the vehicle registry CSV into a table called 'registry'
con.execute("""
    CREATE TABLE registry AS
    SELECT * FROM read_csv_auto('../raw_data/legacy_vehicle_registry.csv')
""")

# Load the payment transaction log CSV into a table called 'payments'
con.execute("""
    CREATE TABLE payments AS
    SELECT * FROM read_csv_auto('../raw_data/payment_transaction_log.csv')
""")

print('✅ Both tables loaded successfully!')
```

**📖 Line-by-line explanation:**
- `duckdb.connect()`: Creates a new in-memory database. Data lives only in RAM — nothing 
  is written to disk. When you close the connection, the data is gone.
- `CREATE TABLE registry AS SELECT * FROM read_csv_auto(...)`: This is DuckDB's powerful 
  CSV reader. `read_csv_auto` automatically detects delimiters, data types, and headers. 
  The result is stored as a named table called `registry`.
- The path `../raw_data/` means 'go up one directory from this notebook, then into raw_data'. 
  This works because your notebook is inside your personal workspace folder (e.g., `./kevin/`).

**💡 Why this matters for DVLA:**
Loading data into memory tables allows us to run SQL queries at high speed without modifying 
the original CSV files. The raw data remains untouched — this is the principle of 
**non-destructive profiling**.

In [ ]:
# Step 2: Create In-Memory Connection and Load CSVs
# Type your code below


## Step 3: Inspect Table Schemas

**📋 What you will do:**
Examine the column names, data types, and nullability of both tables. This is always the 
first step in any data profiling exercise — you need to understand what you're working with.

**🔧 Code to type:**
```python
# Inspect the vehicle registry table structure
print('=== VEHICLE REGISTRY SCHEMA ===')
con.sql('DESCRIBE registry').show()

print()

# Inspect the payment transactions table structure
print('=== PAYMENT TRANSACTIONS SCHEMA ===')
con.sql('DESCRIBE payments').show()
```

**📖 Line-by-line explanation:**
- `DESCRIBE registry`: A SQL command that returns metadata about the table — column names, 
  data types (VARCHAR, DOUBLE, etc.), and whether nulls are allowed. DuckDB's `read_csv_auto` 
  inferred these types automatically.
- `.show()`: Prints the result in a formatted table directly in the Jupyter output cell.

**💡 Why this matters for DVLA:**
Notice that `Registration_Date` may appear as VARCHAR (text) rather than DATE. This is because 
our raw data contains **mixed date formats** (YYYY-MM-DD and DD/MM/YYYY), so DuckDB cannot 
safely cast it to a date type. This is exactly the kind of issue we need to flag.

In [ ]:
# Step 3: Inspect Table Schemas
# Type your code below


## Step 4: Basic Profiling — Row Counts and Distinct Values

**📋 What you will do:**
Count total rows and distinct values in key columns. This gives you a quick health check 
of the data — if total rows differ significantly from distinct rows, you have duplicates.

**🔧 Code to type:**
```python
# Count total and distinct rows in the vehicle registry
con.sql("""
    SELECT
        COUNT(*)                         AS total_rows,
        COUNT(DISTINCT Registration_ID)  AS unique_registrations,
        COUNT(DISTINCT Chassis_Number)    AS unique_chassis,
        COUNT(DISTINCT Owner_Name)        AS unique_owners,
        COUNT(DISTINCT Regional_Office)   AS branch_count
    FROM registry
""").show()

# Count total and distinct rows in the payment log
con.sql("""
    SELECT
        COUNT(*)                         AS total_rows,
        COUNT(DISTINCT Transaction_ID)   AS unique_transactions,
        COUNT(DISTINCT Registration_ID)  AS unique_linked_vehicles,
        COUNT(DISTINCT Payment_Channel)  AS channel_count
    FROM payments
""").show()
```

**📖 Line-by-line explanation:**
- `COUNT(*)`: Counts every row in the table, including duplicates.
- `COUNT(DISTINCT column)`: Counts only unique values. If `COUNT(*)` is 52,500 but 
  `COUNT(DISTINCT Registration_ID)` is 50,000, there are approximately 2,500 duplicate rows.
- `AS alias`: Gives each result column a readable name.

**💡 Why this matters for DVLA:**
If the total row count is significantly higher than the distinct registration count, 
it means the same vehicle may have been registered multiple times — inflating fleet 
statistics and potentially enabling double-billing.

In [ ]:
# Step 4: Basic Profiling
# Type your code below


## Step 5: Identify Exact Duplicate Records

**📋 What you will do:**
Find rows that appear more than once by grouping on all columns. The SQL `HAVING` clause 
filters groups to show only those with a count greater than 1.

**🔧 Code to type:**
```python
# Find exact duplicate vehicle registration records
con.sql("""
    SELECT
        Registration_ID,
        Chassis_Number,
        Owner_Name,
        Regional_Office,
        COUNT(*) AS occurrence_count
    FROM registry
    GROUP BY Registration_ID, Chassis_Number, Owner_Name, Regional_Office
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC
    LIMIT 20
""").show()
```

**📖 Line-by-line explanation:**
- `GROUP BY col1, col2, ...`: Groups all rows that have identical values across the listed 
  columns into a single bucket.
- `HAVING COUNT(*) > 1`: The `HAVING` clause is like `WHERE`, but it filters **after** grouping. 
  It keeps only groups where the count exceeds 1 — i.e., duplicates.
- `ORDER BY occurrence_count DESC`: Shows the most-duplicated records first.
- `LIMIT 20`: Restricts output to the top 20 for readability.

**💡 Why this matters for DVLA:**
Duplicate registrations can occur when the same vehicle is processed at multiple branch 
offices, or when paper records are digitized more than once. Each duplicate inflates the 
fleet count and could result in duplicate fee collections — a revenue integrity issue.

In [ ]:
# Step 5: Duplicate Detection
# Type your code below


## Step 6: Profile Missing National ID Numbers

**📋 What you will do:**
Count how many vehicle registrations have a missing (NULL) Ghana Card number, broken down 
by regional branch office. This reveals which offices have the worst data capture practices.

**🔧 Code to type:**
```python
# Count null National_ID_Number by regional office
con.sql("""
    SELECT
        Regional_Office,
        COUNT(*)                                              AS total_records,
        SUM(CASE WHEN National_ID_Number IS NULL THEN 1 ELSE 0 END) AS null_id_count,
        ROUND(SUM(CASE WHEN National_ID_Number IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS null_percentage
    FROM registry
    GROUP BY Regional_Office
    ORDER BY null_percentage DESC
""").show()
```

**📖 Line-by-line explanation:**
- `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`: This is a conditional count. For each row, if 
  `National_ID_Number IS NULL`, it adds 1; otherwise 0. The SUM gives the total null count.
- `ROUND(... * 100.0 / COUNT(*), 2)`: Calculates the percentage of null IDs and rounds to 
  2 decimal places. The `100.0` (not `100`) forces floating-point division.
- `ORDER BY null_percentage DESC`: Shows the offices with the highest null rates first.

**💡 Why this matters for DVLA:**
A vehicle registration without a verified National ID means the owner cannot be definitively 
identified. This creates risks for law enforcement (stolen vehicles), insurance claims, 
and tax collection. The Ghana Card integration initiative aims to close this gap.

In [ ]:
# Step 6: Null National ID Profiling
# Type your code below


## Step 7: Detect Payment Anomalies — Zero and Negative Amounts

**📋 What you will do:**
Identify payment transactions where the amount paid is zero or negative. These represent 
potential revenue leakage — money that should have been collected but wasn't.

**🔧 Code to type:**
```python
# Find payments with zero or negative amounts
con.sql("""
    SELECT
        Transaction_ID,
        Registration_ID,
        Amount_Paid_GHS,
        Payment_Channel,
        Payment_Timestamp
    FROM payments
    WHERE Amount_Paid_GHS <= 0
    ORDER BY Amount_Paid_GHS ASC
    LIMIT 20
""").show()

# Count total anomalous payments
con.sql("""
    SELECT
        COUNT(*) AS total_anomalous_payments,
        SUM(Amount_Paid_GHS) AS total_revenue_loss_ghs
    FROM payments
    WHERE Amount_Paid_GHS <= 0
""").show()
```

**📖 Line-by-line explanation:**
- `WHERE Amount_Paid_GHS <= 0`: Filters for payments that are zero (no payment received) 
  or negative (potentially fraudulent refund entries).
- `ORDER BY Amount_Paid_GHS ASC`: Shows the most negative (largest loss) values first.
- `SUM(Amount_Paid_GHS)`: When summing negative values, the total represents the 
  aggregate revenue loss. A large negative total is a red flag for the finance team.

**💡 Why this matters for DVLA:**
Zero-amount payments may indicate processing errors at POS terminals. Negative amounts 
could indicate unauthorized refunds. Both must be investigated by the Revenue Assurance unit.

In [ ]:
# Step 7: Payment Anomalies
# Type your code below


## Step 8: Detect Orphan Payments — Unlinked Registration IDs

**📋 What you will do:**
Find payment transactions that reference a Registration_ID which does not exist in the 
vehicle registry. These 'orphan' payments represent money collected for vehicles that have 
no official registration record.

**🔧 Code to type:**
```python
# Find payments with no matching vehicle registration
con.sql("""
    SELECT
        p.Transaction_ID,
        p.Registration_ID,
        p.Amount_Paid_GHS,
        p.Payment_Channel
    FROM payments p
    LEFT JOIN registry r ON p.Registration_ID = r.Registration_ID
    WHERE r.Registration_ID IS NULL
    LIMIT 20
""").show()

# Count total orphan payments and their financial impact
con.sql("""
    SELECT
        COUNT(*)                 AS orphan_payment_count,
        ROUND(SUM(p.Amount_Paid_GHS), 2)  AS orphan_revenue_ghs
    FROM payments p
    LEFT JOIN registry r ON p.Registration_ID = r.Registration_ID
    WHERE r.Registration_ID IS NULL
""").show()
```

**📖 Line-by-line explanation:**
- `LEFT JOIN registry r ON p.Registration_ID = r.Registration_ID`: A LEFT JOIN returns all 
  rows from the left table (payments) and matching rows from the right table (registry). 
  If there's no match, the right-side columns are NULL.
- `WHERE r.Registration_ID IS NULL`: Filters for payments where no matching registration 
  was found — these are the orphan records.
- Table aliases (`p` for payments, `r` for registry) make the query more readable.

**💡 Why this matters for DVLA:**
Orphan payments could indicate: (a) vehicles registered under a different ID format, 
(b) payments processed at unauthorized collection points, or (c) data migration errors 
where the registry record was lost. Each scenario requires different remediation.

In [ ]:
# Step 8: Orphan Payment Detection
# Type your code below


## Step 9: Anomaly Summary by Regional Office

**📋 What you will do:**
Create a comprehensive summary showing all data quality issues aggregated by regional branch 
office. This gives DVLA leadership a clear picture of which offices need the most attention.

**🔧 Code to type:**
```python
# Comprehensive anomaly summary per regional office
con.sql("""
    SELECT
        r.Regional_Office,
        COUNT(DISTINCT r.Registration_ID) AS unique_vehicles,
        SUM(CASE WHEN r.National_ID_Number IS NULL THEN 1 ELSE 0 END) AS missing_ids,
        COUNT(*) - COUNT(DISTINCT r.Registration_ID || r.Chassis_Number || COALESCE(r.Owner_Name, '')) AS approx_duplicates
    FROM registry r
    GROUP BY r.Regional_Office
    ORDER BY missing_ids DESC
""").show()
```

**📖 Line-by-line explanation:**
- `COUNT(DISTINCT r.Registration_ID)`: Counts unique vehicles per office.
- `SUM(CASE WHEN ... IS NULL ...)`: Counts missing National IDs per office.
- `COALESCE(r.Owner_Name, '')`: Replaces NULL owner names with empty string for concatenation.
- The difference between total count and distinct concatenated keys approximates duplicates.

**💡 Why this matters for DVLA:**
This summary becomes the basis for the **Data Cleanup Initiative** action plan — offices 
with the highest anomaly rates should be prioritized for staff retraining and system audits.

In [ ]:
# Step 9: Anomaly Summary by Regional Office
# Type your code below


## Step 10: Payment Channel Distribution Analysis

**📋 What you will do:**
Analyze how payments are distributed across different channels (MTN MoMo, Bank Branch, etc.) 
and identify which channels have the highest rates of anomalous transactions.

**🔧 Code to type:**
```python
# Payment channel distribution with anomaly rates
con.sql("""
    SELECT
        Payment_Channel,
        COUNT(*)                                         AS total_transactions,
        ROUND(SUM(CASE WHEN Amount_Paid_GHS > 0 THEN Amount_Paid_GHS ELSE 0 END), 2) AS valid_revenue_ghs,
        SUM(CASE WHEN Amount_Paid_GHS <= 0 THEN 1 ELSE 0 END)    AS anomalous_count,
        ROUND(SUM(CASE WHEN Amount_Paid_GHS <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS anomaly_rate_pct
    FROM payments
    GROUP BY Payment_Channel
    ORDER BY total_transactions DESC
""").show()
```

**📖 Line-by-line explanation:**
- `SUM(CASE WHEN Amount_Paid_GHS > 0 THEN Amount_Paid_GHS ELSE 0 END)`: Sums only valid 
  (positive) payments, ignoring zero and negative entries.
- `anomaly_rate_pct`: Shows the percentage of anomalous transactions per channel. A high 
  rate on a specific channel may indicate a systemic issue with that payment gateway.

**💡 Why this matters for DVLA:**
If MTN MoMo shows a disproportionately high anomaly rate compared to Bank Branch, it could 
indicate integration issues with the mobile money gateway that need to be escalated to the 
payment service provider.

In [ ]:
# Step 10: Payment Channel Distribution
# Type your code below


## Step 11: Discussion — What Do These Findings Mean?

### Key Findings from Our Profiling

After completing the analysis above, you should have identified:

| Issue | Expected Scale | Impact |
|---|---|---|
| Duplicate registrations | ~5% of records | Inflated fleet statistics, double billing |
| Missing National IDs | ~10% of records | Unverifiable vehicle ownership |
| Mixed date formats | ~30% of records | Breaks automated date-based reporting |
| Zero/negative payments | ~5% of transactions | Direct revenue leakage |
| Orphan payments | ~10% of transactions | Unlinked revenue, potential fraud |

### What Happens Next?

In **Lab 2**, you will use **Apache PySpark** to:

1. **Clean** these issues programmatically at scale
2. **Standardize** date formats across all records
3. **Map** regional offices to the new 2026 Zonal Area Codes
4. **Extract** the latest payment record per vehicle
5. **Export** clean data for Power BI dashboard reporting

### 🎓 Reflection Questions

1. Which regional office has the worst data quality? What might explain this?
2. What is the total revenue at risk from zero/negative payments?
3. How would you prioritize the cleanup — fix duplicates first, or missing IDs?

---
*Lab 1 Complete — Kevin, proceed to Lab 2 when ready.*

## Cleanup: Close the DuckDB Connection

**🔧 Code to type:**
```python
# Always close database connections when finished
con.close()
print('✅ DuckDB connection closed.')
```

**📖 Explanation:**
Closing the connection releases the in-memory tables and frees RAM. This is good practice, 
especially on a shared server where multiple students are working simultaneously.

In [ ]:
# Cleanup: Close the DuckDB Connection
# Type your code below
